# Churn — nettoyage des tables et construction de la table finale

Ce notebook répond à **deux questions** :

1. comment on **nettoie chaque table**, une par une ;
2. comment on **les réunit en une seule table**, prête pour le modèle.

**Entrée** : 6 fichiers CSV dans `data/raw/`
**Sortie** : `data/final/all_tables.csv` — 34 000 lignes × 10 colonnes

| Fichier source | Contenu | Grain |
|---|---|---|
| `CIBLE_CHURN_SAMPLE2_new.csv` | profil du client + label | 1 ligne / contrat |
| `DATA_...csv` | consommation internet | 1 ligne / contrat / mois |
| `ENTRANT_...csv` | appels et SMS **reçus** | 1 ligne / contrat / mois |
| `SORTANT_...csv` | appels et SMS **émis** | 1 ligne / contrat / mois |
| `RECHARGE_...csv` | recharges | 1 ligne / contrat / mois |
| `USSD_...csv` | achats de forfaits | 1 ligne / contrat / mois |

## 0. Préparation

Deux réflexes indispensables sur ces fichiers :

- le séparateur est `;` et **les valeurs ont des espaces autour** → `strip()` obligatoire, sinon les jointures échouent silencieusement ;
- `MONTH_DT` est une date Oracle : `01JAN2025:00:00:00`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# dossiers du projet
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
FINAL = ROOT / 'data' / 'final'
for d in (PROCESSED, FINAL):
    d.mkdir(parents=True, exist_ok=True)

FICHIERS = {
    'cible':    'CIBLE_CHURN_SAMPLE2_new.csv',
    'data':     'DATA_CHURN_2NW_2025.csv',
    'entrant':  'ENTRANT_CHURN_2NW_2025.csv',
    'sortant':  'SORTANT_CHURN_2NW_2025.csv',
    'recharge': 'RECHARGE_CHURN_2NW_2025.csv',
    'ussd':     'USSD_CHURN_2NW_2025.csv',
}

def charger(nom):
    """Lit un fichier brut et enlève les espaces autour des noms et des valeurs."""
    df = pd.read_csv(RAW / FICHIERS[nom], sep=';', dtype=str, keep_default_na=False)
    df.columns = [c.strip() for c in df.columns]
    for c in df.columns:
        df[c] = df[c].str.strip()
    return df

print('Racine du projet :', ROOT)
print('Fichiers déclarés :', len(FICHIERS))

Racine du projet : /Users/mohamedyassinchannoufi/Desktop/ML churn
Fichiers déclarés : 6


## 1. Comprendre les clés avant de toucher aux données

Une seule clé relie tout le monde : **`CODE_CONTRAT`**.

| Table | Grain | Clé |
|---|---|---|
| `cible` | 1 ligne / contrat | `CODE_CONTRAT` |
| les 5 autres | 1 ligne / contrat **et par mois** | `CODE_CONTRAT` + `MONTH_DT` |

**Règle d'or** : on ne joint jamais deux tables de faits entre elles.
12 lignes × 12 lignes donneraient 144 lignes par contrat.

La méthode en deux temps :

```
1. réduire chaque table de faits à 1 ligne par contrat   (agrégation)
2. joindre les résultats à la table cible                (left join)
```

In [2]:
# contrôle des clés : unicité et couverture
base = set(charger('cible')['CODE_CONTRAT'])
print(f'contrats dans la cible : {len(base):,}')
print()

for nom in ['data', 'entrant', 'sortant', 'recharge', 'ussd']:
    df = charger(nom)
    dup = df.duplicated(subset=['CODE_CONTRAT', 'MONTH_DT']).sum()
    ids = set(df['CODE_CONTRAT'])
    print(f'{nom:9s} {len(df):>7,} lignes | doublons (contrat,mois) : {dup} | '
          f'absents de la cible : {len(ids - base)}')

contrats dans la cible : 34,000



data      164,563 lignes | doublons (contrat,mois) : 0 | absents de la cible : 0


entrant   352,982 lignes | doublons (contrat,mois) : 0 | absents de la cible : 0


sortant   330,000 lignes | doublons (contrat,mois) : 0 | absents de la cible : 0


recharge  282,241 lignes | doublons (contrat,mois) : 0 | absents de la cible : 0
ussd      145,793 lignes | doublons (contrat,mois) : 0 | absents de la cible : 0


## 2. Nettoyage, table par table

Chaque table a ses propres problèmes. Voici les décisions, une table à la fois.

---

### 2.1 Table `cible` — le profil du client

8 colonnes au départ, 7 à la sortie.

| Colonne | Décision | Raison |
|---|---|---|
| `CODE_CONTRAT` | **gardée** | c'est la clé de jointure |
| `ANC_J` | **supprimée** | doublon exact de `ANC_M` (×30,4) |
| `ANC_M` | gardée | ancienneté en mois |
| `HANDSET` | gardée | signal fort (`NA` = 57 % de churn) |
| `Canal_De_Vente` | **supprimée** | redondante avec `ANC_M` (mesuré +0,002 d'AUC) |
| `OFFRE` | gardée | nom de l'offre |
| `STATUT` | gardée | état actuel (A / S) |
| `statut_next_month` | **remplacée par `churn`** (0/1) | c'est le label |

**Règles de calcul du label** :

- `churn = 1` si le client part (`D`) ou est suspendu (`S`)
- `churn = 0` s'il reste actif (`A`)

In [3]:
# --- nettoyage de la table cible ---
cible = charger('cible')

# 1. le label, calculé AVANT de perdre statut_next_month
cible['churn'] = (cible['statut_next_month'] != 'A').astype(int)

# 2. on garde seulement les colonnes utiles, dans l'ordre voulu
COLONNES_CIBLE = ['CODE_CONTRAT', 'ANC_M', 'HANDSET', 'OFFRE', 'STATUT', 'churn']
cible = cible[COLONNES_CIBLE].copy()

# 3. ANC_M en nombre
cible['ANC_M'] = pd.to_numeric(cible['ANC_M'])

print('cible nettoyée :', cible.shape)
print('colonnes       :', list(cible.columns))
print('taux de churn  :', round(cible['churn'].mean() * 100, 2), '%')
print()
print(cible.head(3).to_string(index=False))

cible nettoyée : (34000, 6)
colonnes       : ['CODE_CONTRAT', 'ANC_M', 'HANDSET', 'OFFRE', 'STATUT', 'churn']
taux de churn  : 28.12 %

CODE_CONTRAT  ANC_M HANDSET          OFFRE STATUT  churn
 SN_10004606    177      4G PRE - TT 1500%      A      0
 SN_10006819    177      2G   PRE - Double      A      0
 SN_10025174    178      4G PRE - TT 1500%      A      0


---

### 2.2 Les 5 tables mensuelles — le même traitement de base

Toutes les tables de faits partagent **trois étapes identiques** :

| Étape | Pourquoi |
|---|---|
| **1. Supprimer les colonnes redondantes** | agrégats, doublons, constantes |
| **2. Supprimer les lignes à `MONTH_DT` vide** | ces clients n'ont aucune activité sur l'année ; toutes leurs valeurs valent 0 |
| **3. Exclure décembre** | le label parle du mois suivant → décembre contient déjà la réponse (fuite) |

### 2.3 Preuve que décembre doit être exclu

In [4]:
# en décembre, les clients partis n'ont presque plus d'activité : le label fuit
s = charger('sortant')
s['mois'] = pd.to_datetime(s['MONTH_DT'], format='%d%b%Y:%H:%M:%S', errors='coerce')
s['duree'] = pd.to_numeric(s['DUREE_APPEL_TOT'])
actif = s.dropna(subset=['mois'])

pivot = actif.pivot_table(index='CODE_CONTRAT', columns=actif['mois'].dt.month,
                          values='duree', aggfunc='sum')
label = cible.set_index('CODE_CONTRAT')['churn']

for mois in [11, 12]:
    jointure = label.to_frame('churn').join(pivot[mois].rename('duree'))
    taux = jointure.groupby('churn')['duree'].apply(lambda v: round(v.isna().mean() * 100, 1))
    print(f'mois {mois} — % de clients SANS aucun appel sortant')
    print(f'   restent (churn=0) : {taux.get(0)} %')
    print(f'   partent (churn=1) : {taux.get(1)} %')
    print()
print('En décembre le label se lit dans l absence de données.')
print('La fenêtre d observation doit donc s arrêter en novembre.')

mois 11 — % de clients SANS aucun appel sortant
   restent (churn=0) : 7.1 %
   partent (churn=1) : 33.9 %

mois 12 — % de clients SANS aucun appel sortant
   restent (churn=0) : 6.0 %
   partent (churn=1) : 63.6 %

En décembre le label se lit dans l absence de données.
La fenêtre d observation doit donc s arrêter en novembre.


### 2.4 Ce qu'on garde dans chaque table

Règle : **un seul axe métier**, toujours un **total**, jamais une composante.

| Table | Colonne gardée | Sens | Pourquoi |
|---|---|---|---|
| `data` | `VOLUME_SESSION` | internet | seule mesure de la table |
| `entrant` | `DUREE_APPEL_IN` | appels **reçus** | c'est déjà le total des 5 origines |
| `entrant` | `NB_SMS_IN` | SMS **reçus** | c'est déjà le total |
| `sortant` | `DUREE_APPEL_TOT` | appels **émis** | c'est déjà le total (on-net + off-net) |
| `sortant` | `nb_sms_tot` | SMS **émis** | seule mesure de ce type |
| `recharge` | `MNT_RECH` | argent rechargé | le montant compte, pas le nombre d'opérations |
| `ussd` | `MNT_FORFAIT` | argent en forfaits | idem |

**Pourquoi pas les colonnes détaillées ?**

```
DUREE_APPEL_IN = DUREE_TT_GSM_IN + DUREE_TT_FIXE_IN
               + DUREE_OOREDOO_IN + DUREE_ORANGE_IN
               + DUREE_APPEL_INTER_IN
```

`DUREE_APPEL_IN` **contient déjà** tout le reste. Les additionner compterait deux fois l'off-net.

In [5]:
# colonnes à supprimer, table par table
A_SUPPRIMER = {
    'recharge': ['STATUT'],                      # constant sur toute l'année
    'sortant':  ['Duree_offnet_tot',             # = somme des 4 opérateurs
                 'revenu_inter', 'revenu_roaming',
                 'revenu_inter_roaming',
                 'DUREE_APPEL_LYCA_TOT', 'DUREE_APPEL_NESSMA_TOT'],
    'entrant':  ['DUREE_OFFNET_IN', 'NB_SMS_OFFNET_IN',   # = Ooredoo + Orange
                 'DUREE_TT_FIXE_IN', 'DUREE_APPEL_INTER_IN'],
    'data':     [],
    'ussd':     [],
}

# colonnes gardées après agrégation
MESURES = {
    'data':     ['VOLUME_SESSION'],
    'entrant':  ['DUREE_APPEL_IN', 'NB_SMS_IN'],
    'sortant':  ['DUREE_APPEL_TOT', 'nb_sms_tot'],
    'recharge': ['MNT_RECH'],
    'ussd':     ['MNT_FORFAIT'],
}
print('configuration prête')

configuration prête


### 2.5 La fonction de nettoyage commune

Elle applique les 3 étapes identiques à toutes les tables mensuelles.

In [6]:
def nettoyer_mensuelle(nom, dernier_mois=11):
    """Nettoie une table mensuelle (1 ligne par contrat et par mois).

    - retire les colonnes inutiles
    - retire les lignes sans date (client inactif toute l'année)
    - retire les mois hors fenêtre (décembre exclu par défaut)
    - convertit les colonnes de chiffres
    """
    df = charger(nom)
    avant = len(df)

    # étape 1 : colonnes inutiles
    df = df.drop(columns=[c for c in A_SUPPRIMER.get(nom, []) if c in df.columns])

    # étape 2 : lignes sans date
    df['mois'] = pd.to_datetime(df['MONTH_DT'], format='%d%b%Y:%H:%M:%S', errors='coerce')
    df = df.dropna(subset=['mois'])

    # étape 3 : fenêtre d'observation
    df = df[df['mois'].dt.month <= dernier_mois].copy()

    # étape 4 : typage numérique
    for c in df.columns:
        if c in ('CODE_CONTRAT', 'MONTH_DT', 'mois'):
            continue
        df[c] = pd.to_numeric(df[c], errors='coerce')

    print(f'{nom:9s} {avant:>7,} -> {len(df):>7,} lignes  ({avant - len(df):>6,} supprimées)')
    return df

tables = {}
for nom in ['data', 'entrant', 'sortant', 'recharge', 'ussd']:
    tables[nom] = nettoyer_mensuelle(nom)

data      164,563 -> 143,210 lignes  (21,353 supprimées)


entrant   352,982 -> 322,965 lignes  (30,017 supprimées)


sortant   330,000 -> 303,468 lignes  (26,532 supprimées)


recharge  282,241 -> 259,682 lignes  (22,559 supprimées)


ussd      145,793 -> 125,110 lignes  (20,683 supprimées)


In [7]:
# contrôle : plus aucune date vide, décembre a disparu
for nom, df in tables.items():
    m_min = int(df['mois'].dt.month.min())
    m_max = int(df['mois'].dt.month.max())
    print(f'{nom:9s} mois de {m_min} à {m_max} | dates vides : {df["mois"].isna().sum()}')

data      mois de 1 à 11 | dates vides : 0
entrant   mois de 1 à 11 | dates vides : 0
sortant   mois de 1 à 11 | dates vides : 0
recharge  mois de 1 à 11 | dates vides : 0
ussd      mois de 1 à 11 | dates vides : 0


## 3. Agrégation : de 12 lignes à 1 ligne par contrat

Pour chaque mesure, on calcule une valeur unique par contrat. `MONTH_DT` disparaît.

**Deux calculs différents**, selon la nature de la mesure :

| Mesure | Calcul | Pourquoi |
|---|---|---|
| `VOLUME_SESSION`, `MNT_RECH`, `MNT_FORFAIT` | **moyenne** mensuelle | représente un niveau d'usage par mois |
| appels et SMS totaux | **somme** sur les 11 mois | représente un volume cumulé |

Pour les appels et SMS, on additionne **reçu + émis mois par mois**, puis on fait la somme :

```
APPEL_TOTAL = Σ (DUREE_APPEL_IN + DUREE_APPEL_TOT)  sur 11 mois
SMS_TOTAL   = Σ (NB_SMS_IN + nb_sms_tot)            sur 11 mois
```

C'est différent (et plus juste) que de faire la moyenne des deux moyennes : un client qui reçoit en janvier et émet en février a bien **2 mois** d'activité.

In [8]:
# --- moyennes mensuelles (niveau d'usage) ---
blocs = {}
for nom in ['data', 'recharge', 'ussd']:
    blocs[nom] = tables[nom].groupby('CODE_CONTRAT')[MESURES[nom]].mean()
    print(f'{nom:9s} -> {len(blocs[nom]):>6,} contrats | moyenne de {list(blocs[nom].columns)}')

# --- sommes pour les appels et SMS (volume cumulé) ---
recu = tables['entrant'][['CODE_CONTRAT', 'mois', 'DUREE_APPEL_IN', 'NB_SMS_IN']]
emis = tables['sortant'][['CODE_CONTRAT', 'mois', 'DUREE_APPEL_TOT', 'nb_sms_tot']]

mensuel = recu.merge(emis, on=['CODE_CONTRAT', 'mois'], how='outer')
mensuel['APPEL_TOTAL'] = mensuel['DUREE_APPEL_IN'].fillna(0) + mensuel['DUREE_APPEL_TOT'].fillna(0)
mensuel['SMS_TOTAL'] = mensuel['NB_SMS_IN'].fillna(0) + mensuel['nb_sms_tot'].fillna(0)

blocs['appels'] = mensuel.groupby('CODE_CONTRAT')[['APPEL_TOTAL', 'SMS_TOTAL']].sum()
print(f'{"appels":9s} -> {len(blocs["appels"]):>6,} contrats | somme de {list(blocs["appels"].columns)}')

# --- la cible est déjà à 1 ligne par contrat ---
blocs['cible'] = cible.set_index('CODE_CONTRAT')
print(f'{"cible":9s} -> {len(blocs["cible"]):>6,} contrats | {list(blocs["cible"].columns)}')

data      -> 24,271 contrats | moyenne de ['VOLUME_SESSION']
recharge  -> 33,783 contrats | moyenne de ['MNT_RECH']
ussd      -> 23,372 contrats | moyenne de ['MNT_FORFAIT']


appels    -> 33,972 contrats | somme de ['APPEL_TOTAL', 'SMS_TOTAL']
cible     -> 34,000 contrats | ['ANC_M', 'HANDSET', 'OFFRE', 'STATUT', 'churn']


### Vérification sur un contrat réel

On prend `SN_10004606` et on recalcule sa moyenne data **et** sa somme d'appels à la main.

In [9]:
sn = 'SN_10004606'

# moyenne data
b = charger('data')
b = b[b['CODE_CONTRAT'] == sn].copy()
b['mois'] = pd.to_datetime(b['MONTH_DT'], format='%d%b%Y:%H:%M:%S', errors='coerce')
b = b.dropna(subset=['mois'])
b = b[b['mois'].dt.month <= 11]
b['VOLUME_SESSION'] = pd.to_numeric(b['VOLUME_SESSION'])

print(f'--- moyenne data ({len(b)} mois) ---')
print('recalculée ici :', round(b['VOLUME_SESSION'].mean(), 2))
print('dans le bloc   :', round(blocs['data'].loc[sn, 'VOLUME_SESSION'], 2))

# somme appels
e = charger('entrant')
e = e[e['CODE_CONTRAT'] == sn].copy()
e['mois'] = pd.to_datetime(e['MONTH_DT'], format='%d%b%Y:%H:%M:%S', errors='coerce')
e = e.dropna(subset=['mois']); e = e[e['mois'].dt.month <= 11]
e['DUREE_APPEL_IN'] = pd.to_numeric(e['DUREE_APPEL_IN'])

s = charger('sortant')
s = s[s['CODE_CONTRAT'] == sn].copy()
s['mois'] = pd.to_datetime(s['MONTH_DT'], format='%d%b%Y:%H:%M:%S', errors='coerce')
s = s.dropna(subset=['mois']); s = s[s['mois'].dt.month <= 11]
s['DUREE_APPEL_TOT'] = pd.to_numeric(s['DUREE_APPEL_TOT'])

total = (e['DUREE_APPEL_IN'].sum() + s['DUREE_APPEL_TOT'].sum())
print(f'--- somme appels ---')
print('recalculée ici :', round(total, 2))
print('dans le bloc   :', round(blocs['appels'].loc[sn, 'APPEL_TOTAL'], 2))

--- moyenne data (11 mois) ---
recalculée ici : 5277.1
dans le bloc   : 5277.1


--- somme appels ---
recalculée ici : 7359.43
dans le bloc   : 7359.43


In [10]:
# sauvegarde des blocs intermédiaires
for nom, bloc in blocs.items():
    bloc.reset_index().to_csv(PROCESSED / f'agg_{nom}.csv', index=False)

print('fichiers écrits dans', PROCESSED)
for f in sorted(PROCESSED.glob('agg_*.csv')):
    d = pd.read_csv(f, nrows=1, keep_default_na=False)
    print(f'  {f.name:20s} {len(d.columns)} colonnes')

fichiers écrits dans /Users/mohamedyassinchannoufi/Desktop/ML churn/data/processed
  agg_appels.csv       3 colonnes
  agg_cible.csv        6 colonnes
  agg_data.csv         2 colonnes
  agg_entrant.csv      3 colonnes
  agg_recharge.csv     2 colonnes
  agg_sortant.csv      3 colonnes
  agg_ussd.csv         2 colonnes


## 4. Réunir les tables en une seule

C'est la question centrale : **comment mettre 6 tables dans 1 ?**

### Le principe

```
        cible  (34 000 contrats)   ← la base, elle porte le label
          │
          │  left join sur CODE_CONTRAT
          ├────────┬────────┬────────┐
          ▼        ▼        ▼        ▼
        data  recharge  ussd    appels
```

On part de la cible et on ajoute les blocs **un par un** avec `how='left'`.

### Pourquoi `left` et pas `inner` ?

Un `inner join` ne garderait que les contrats présents dans **toutes** les tables. Or 28,6 % des clients n'ont aucune session data, et 31,3 % n'achètent aucun forfait. On perdrait un tiers de la base.

Avec `left`, on garde les 34 000 contrats. Ceux qui n'ont pas d'activité dans une table gardent une case vide : c'est une information en soi.

In [11]:
# on part de la cible
table_finale = blocs['cible'].copy()
print('départ :', table_finale.shape)

# on ajoute les blocs un par un
for nom in ['data', 'recharge', 'ussd', 'appels']:
    table_finale = table_finale.join(blocs[nom], how='left')
    print(f'+ {nom:9s} -> {table_finale.shape[0]:,} lignes x {table_finale.shape[1]} colonnes')

print()
print('forme finale :', table_finale.shape)

# le contrôle se fait AVANT de retirer la clé
assert len(table_finale) == 34000, 'la jointure a dupliqué des lignes !'
print('contrôle : 34 000 lignes, 1 par contrat ✓')

départ : (34000, 5)
+ data      -> 34,000 lignes x 6 colonnes
+ recharge  -> 34,000 lignes x 7 colonnes
+ ussd      -> 34,000 lignes x 8 colonnes
+ appels    -> 34,000 lignes x 10 colonnes

forme finale : (34000, 10)
contrôle : 34 000 lignes, 1 par contrat ✓


### Ce que signifient les cases vides

Après la jointure, certaines colonnes ont des `NaN`. Ce n'est pas une erreur.

| Colonne | Vides | Signification |
|---|---|---|
| `VOLUME_SESSION` | 28,6 % | aucune session data de l'année |
| `MNT_FORFAIT` | 31,3 % | aucun forfait acheté |
| `APPEL_TOTAL` | 0,1 % | aucun appel reçu ni émis |

**Vide ≠ zéro** : un client sans data n'est pas un client qui consomme 0 Mo. Ces deux situations sont différentes, et le vide est en soi un signal de churn.

Une exception : pour les sommes d'appels et de SMS, un client absent n'a pas une somme inconnue — il a une somme de **0**. On remplit donc ces deux colonnes.

In [12]:
# les sommes absentes valent 0 (aucune activité), pas NaN
table_finale = table_finale.fillna({'APPEL_TOTAL': 0, 'SMS_TOTAL': 0})

print('contrats sans activité, par bloc :')
for nom, cols in [('data', ['VOLUME_SESSION']),
                  ('recharge', ['MNT_RECH']),
                  ('ussd', ['MNT_FORFAIT'])]:
    n = table_finale[cols].isna().all(axis=1).sum()
    print(f'  {nom:9s} {n:>6,}  ({n / len(table_finale) * 100:.1f} %)')

print()
print('sommes à zéro :')
for c in ['APPEL_TOTAL', 'SMS_TOTAL']:
    print(f'  {c:12s} {int((table_finale[c] == 0).sum()):>6,}')

contrats sans activité, par bloc :
  data       9,729  (28.6 %)
  recharge     217  (0.6 %)
  ussd      10,628  (31.3 %)

sommes à zéro :
  APPEL_TOTAL     172
  SMS_TOTAL       123


## 5. Export de la table finale

On retire `CODE_CONTRAT` (c'est la clé, pas une feature) et on écrit le fichier.

Le résultat est directement utilisable comme matrice `X` + `y` : plus rien à retirer avant l'entraînement.

In [13]:
sortie = table_finale.reset_index().drop(columns=['CODE_CONTRAT'])
sortie.to_csv(FINAL / 'all_tables.csv', index=False)

print('fichier écrit :', FINAL / 'all_tables.csv')
print('forme         :', sortie.shape)
print('colonnes      :', list(sortie.columns))
print()
print(sortie.head(3).to_string(index=False))

fichier écrit : /Users/mohamedyassinchannoufi/Desktop/ML churn/data/final/all_tables.csv
forme         : (34000, 10)
colonnes      : ['ANC_M', 'HANDSET', 'OFFRE', 'STATUT', 'churn', 'VOLUME_SESSION', 'MNT_RECH', 'MNT_FORFAIT', 'APPEL_TOTAL', 'SMS_TOTAL']

 ANC_M HANDSET          OFFRE STATUT  churn  VOLUME_SESSION  MNT_RECH  MNT_FORFAIT  APPEL_TOTAL  SMS_TOTAL
   177      4G PRE - TT 1500%      A      0     5277.095455 21.727273    14.281818      7359.43     1768.0
   177      2G   PRE - Double      A      0             NaN  5.222222     0.800000       598.95      592.0
   178      4G PRE - TT 1500%      A      0     5242.772727 27.727273     1.737500     10223.95     2290.0


## 6. Récapitulatif

### Le nettoyage, table par table

| Table | Colonnes supprimées | Lignes supprimées | Résultat |
|---|---|---|---|
| `cible` | `ANC_J`, `Canal_De_Vente`, `statut_next_month` | aucune | 34 000 lignes, 5 colonnes + `churn` |
| `data` | aucune | vides + décembre | moyenne de `VOLUME_SESSION` |
| `entrant` | 4 (offnet, fixe, inter) | vides + décembre | intègre le total appels/SMS reçus |
| `sortant` | 6 (dont `Duree_offnet_tot`) | vides + décembre | intègre le total appels/SMS émis |
| `recharge` | `STATUT` | vides + décembre | moyenne de `MNT_RECH` |
| `ussd` | aucune | vides + décembre | moyenne de `MNT_FORFAIT` |

### Les 3 règles de nettoyage

1. **Une seule clé** : `CODE_CONTRAT`. Jamais de jointure entre deux tables de faits.
2. **Supprimer les composantes, garder les totaux** : `DUREE_APPEL_IN` contient déjà ses 5 morceaux.
3. **Exclure décembre** : le label parle du mois suivant, donc décembre le contient déjà.

### La fusion en une seule table

```
6 fichiers bruts
      ↓  nettoyage (colonnes, lignes vides, décembre)
5 tables mensuelles + 1 table cible
      ↓  agrégation (moyenne ou somme, 1 ligne par contrat)
6 blocs
      ↓  left join depuis la cible
34 000 lignes × 10 colonnes
      ↓
data/final/all_tables.csv
```

### Prochaine étape

La table est prête, mais deux points restent à traiter avant le modèle :

- **les `NaN`** : les laisser, les remplir à 0, ou ajouter une colonne « a eu de l'activité » ;
- **la validation temporelle** : entraîner sur janvier→juin, tester sur juillet→novembre. Une validation croisée aléatoire surestime la performance sur des données mensuelles.